# TBP — Primera prueba con series reales de Argentina

Este notebook calcula el TBP por cohorte (1950–2035) con las series de `datos/procesados/` (ver `datos/README.md`). Es una **prueba exploratoria**: sirve para ver si los datos se comportan, no para concluir.

**Qué es dato y qué es supuesto** (todo queda a la vista):

| Variable | Valor | Origen | Tipo |
|---|---|---|---|
| $N_t$ | nacimientos | DEIS (≤2024); ONU proyección (2025+) | observado / proyectado |
| $1-\mu$ | supervivencia de la **cohorte** hasta 65 | ONU WPP 2024, tabla de mortalidad (producto de $p_x$ por edad y año calendario) | estimado |
| $J_{t+65}$ | población 65+ en $t+65$ | ONU WPP 2024 | proyectado |
| $E_r$ | esperanza de vida a los 65 en $t+65$ | ONU WPP 2024 (de período) | proyectado |
| $\tau$ | 11 % personal + 10,77 % patronal = **21,77 %** | Anuario ANSES 2008–2023, p. 39 | normativa (sin aportes a PAMI/obra social) |
| $\rho$ | **40,3 %** (haber medio SIPA / salario imponible medio, dic-2023) | Anuario ANSES, p. 91 | observado, **constante** |
| $\bar A$ | **14,2** (obs.) o **30** (legal) años | ANSES p. 46 (varones 64 años: 15,8; mujeres 59 años: 12,6; promedio simple) / requisito legal | **supuesto — escenario** |

> ⚠️ $\bar A$ observado está **sesgado a la baja**: ANSES solo cuenta aportes desde jul-1994 y a personas con ≥12 meses de aporte. El 30 es el requisito legal (techo). La verdad está entre ambos. **Por eso el nivel del TBP no es interpretable todavía; la forma de la curva y su tendencia sí.**

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
try: display
except NameError: display = print

# --- Carga de datos -------------------------------------------------------
# Orden de busqueda: 1) carpeta del repo (../datos/procesados), 2) Drive (opcional), 3) descarga desde GitHub.
# En Colab con Drive: descomentar las dos lineas siguientes para montarlo.
# from google.colab import drive; drive.mount('/content/drive')
ARCHIVOS = ["series_tbp_extraidas.csv", "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"]
# Repositorio del proyecto
RAW_BASE = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDIDATOS = ["../datos/procesados", "datos/procesados",
              "/content/drive/MyDrive/JUANDE/CURSAR/06. Taller de indicadores/datos_tbp",
              "/content/drive/MyDrive/datos_tbp", "/content/datos_tbp", "."]
DATA = next((p for p in CANDIDATOS if os.path.exists(os.path.join(p, "series_tbp_extraidas.csv"))), None)
if DATA is None:   # no hay copia local: descargar desde GitHub (raw)
    DATA = "datos_tbp_descargados"; os.makedirs(DATA, exist_ok=True)
    for f in ARCHIVOS:
        urllib.request.urlretrieve(f"{RAW_BASE}/{f}", os.path.join(DATA, f))
print("Datos en:", os.path.abspath(DATA))

# Las salidas (CSV y PNG) se escriben en ./salidas_tbp para no pisar los datos versionados.
OUT = "salidas_tbp"; os.makedirs(OUT, exist_ok=True)

s = pd.read_csv(f"{DATA}/series_tbp_extraidas.csv").set_index("year")
mort = pd.read_csv(f"{DATA}/un_wpp2024_argentina_tabla_mortalidad_abreviada.csv", low_memory=False)
mort = mort[mort.Sex == "Total"]
display(s.loc[[1950, 1990, 2010, 2024, 2040, 2091], ["deis_nacimientos", "un_births_miles", "un_e65", "un_pob_65mas_miles"]])

## 1. Supervivencia de la cohorte (1−μ)

Para la cohorte nacida en $t$ se multiplica la probabilidad de sobrevivir cada grupo de edad ($0, 1\text{–}4, 5\text{–}9,\dots,60\text{–}64$) usando la tabla de mortalidad del **año calendario en que ese grupo vive**. Es más correcto que usar la tabla del año de nacimiento ("de período"), que ignora que la mortalidad va bajando.

In [ ]:
px = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
grp = mort[mort.Time == 2000].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]]
grp = grp[grp.AgeGrpStart < 65]

def surv65_cohorte(t):
    S = 1.0
    for a, sp in grp.itertuples(index=False):
        y = int(np.floor(t + a + sp/2))
        if y > px.index.max() or y < px.index.min():
            return np.nan
        S *= px.loc[y, a]
    return S

print("Ejemplo cohorte 2000: 1-mu =", round(surv65_cohorte(2000), 4), "| tabla de período del año 2000 (ignora mejoras futuras):", round(s.un_S65_periodo[2000], 4))

## 2. Cálculo por cohorte

$\tau$ y $\rho$ son constantes (los datos de ANSES no dan serie anual utilizable, ver `anses_valores_extraidos.csv`). **Todo el movimiento de la curva viene de la demografía** ($N_t$, $1-\mu$, $J$, $E_r$).

In [ ]:
TAU = 0.11 + 0.1077     # Anuario ANSES p.39 (aporte personal 11% + contribución patronal SIPA 10,77% actividades inc. a)
RHO = 0.403             # Anuario ANSES p.91: haber medio SIPA / salario imponible medio, dic-2023
A_OBS, A_LEGAL = 14.2, 30.0

rows = []
for t in range(1950, 2036):
    y = t + 65
    N_deis = s.deis_nacimientos.get(t, np.nan)
    N_un = s.un_births_miles[t] * 1000
    rows.append(dict(cohorte=t, anio_65=y,
        N_t=N_deis if not np.isnan(N_deis) else N_un,
        fuente_N="DEIS" if not np.isnan(N_deis) else "ONU proyectado",
        N_un=N_un, S65=surv65_cohorte(t),
        J=s.un_pob_65mas_miles[y]*1000, E65=s.un_e65[y], pob_edad65=s.un_pob_edad65_miles[y]*1000))
d = pd.DataFrame(rows)

for nom, A in [("obs", A_OBS), ("legal", A_LEGAL)]:
    d[f"TBP_A_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*d.E65*RHO)     # fórmula original del grupo
    d[f"TBP_C_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*RHO)           # en años (sin E_r)
    d[f"TBP_B_{nom}"] = A*TAU/(d.E65*RHO)                     # cohorte pura, sin N
d["D"] = d.N_t*d.S65/d.J                                       # factor demográfico

d.to_csv(f"{OUT}/tbp_argentina_cohortes_1950_2035.csv", index=False)
display(d[d.cohorte.isin([1950,1975,1990,2000,2010,2015,2020,2024,2030,2035])][
    ["cohorte","N_t","fuente_N","S65","J","E65","D","TBP_A_obs","TBP_A_legal","TBP_C_legal","TBP_B_legal"]])

## 3. Grupo de control con datos reales

Tres chequeos que **no dependen** de los supuestos de $\bar A$, $\tau$ ni $\rho$:

1. **Identidades algebraicas:** $TBP_A = D\cdot B$ y $TBP_A = TBP_C/E_r$.
2. **Consistencia demográfica:** la población de exactamente 65 años en $t+65$ (ONU) debería ser ≈ $N_t \cdot (1-\mu)$ (nacidos que sobreviven; la diferencia es migración y error de las fuentes).
3. **Consistencia entre fuentes:** nacimientos DEIS vs ONU.

In [ ]:
# 1) identidades
B = d.TBP_B_legal
assert np.allclose(d.TBP_A_legal, d.D * B)
assert np.allclose(d.TBP_A_legal, d.TBP_C_legal / d.E65)
print("✅ Identidades TBP_A = D×B y TBP_A = TBP_C/E_r: OK en las 86 cohortes")

# 2) cohorte reconstruida vs población observada/proyectada a los 65
d["ratio_pob65_vs_NS"] = d.pob_edad65 / (d.N_t * d.S65)
r = d.ratio_pob65_vs_NS
print(f"\n2) población(65 años en t+65) / (N_t × S65): media {r.mean():.3f} | min {r.min():.3f} | max {r.max():.3f}")
fuera = d[(r < 0.85) | (r > 1.20)][["cohorte", "fuente_N", "ratio_pob65_vs_NS"]]
print("   Cohortes fuera de [0.85, 1.20]:", "ninguna" if fuera.empty else "")
if not fuera.empty: display(fuera)

# 3) DEIS vs ONU
c = d[d.fuente_N == "DEIS"].copy()
c["ONU/DEIS"] = c.N_un / c.N_t
print(f"\n3) Nacimientos ONU/DEIS ({c.cohorte.min()}–{c.cohorte.max()}): media {c['ONU/DEIS'].mean():.3f} | min {c['ONU/DEIS'].min():.3f} | max {c['ONU/DEIS'].max():.3f}")
display(c.loc[c.cohorte.isin([1950,1980,2000,2015,2020,2022,2023,2024]), ["cohorte", "N_t", "N_un", "ONU/DEIS"]])

## 4. Gráficos

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(d.cohorte, d.N_t/1e3, label="Usado (DEIS ≤2024, ONU después)")
ax[0].plot(d.cohorte, d.N_un/1e3, "--", label="ONU WPP 2024")
ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Nacimientos (miles)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(d.cohorte, d.D, color="tab:green")
ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("D = N·(1−μ) / J₆₅₊ (factor demográfico)"); ax[1].grid(alpha=.3)

ax[2].plot(d.cohorte, d.TBP_A_legal, label="TBP_A  (Ā = 30, techo)")
ax[2].plot(d.cohorte, d.TBP_A_obs, label="TBP_A  (Ā = 14,2, piso sesgado)")
ax[2].plot(d.cohorte, d.TBP_B_legal, "--", label="TBP_B (cohorte pura, Ā = 30)")
ax[2].axhline(1, color="gray", ls=":"); ax[2].axvline(2024.5, color="gray", ls=":")
ax[2].set_title("TBP por cohorte"); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel("año de nacimiento de la cohorte")
plt.tight_layout(); plt.savefig(f"{OUT}/tbp_argentina_graficos.png", dpi=130); plt.show()

## 5. Qué muestran los datos (lectura preliminar)

* **Caída de la natalidad real:** de ~756 mil (2010) a 413 mil (2024). La cohorte 2024 tiene el factor demográfico $D$ más bajo de toda la serie.
* **Rebote artificial después de 2024:** la ONU proyecta ~507 mil nacimientos en 2024 y ~520 mil hacia 2030. Eso **no refleja** la caída registrada por DEIS. El repunte de las cohortes 2025–2035 es un efecto del supuesto ONU, **no un dato**: hay que reemplazarlo con un escenario propio de natalidad.
* **Longevidad:** $E_{65}$ pasa de 17,9 (cohorte 1950) a ~24,6 (cohorte 2035). Eso empuja el TBP hacia abajo por dos vías (en $J$ y en $E_r$), el problema de doble conteo detectado en la auditoría.
* **La curva de $TBP_B$** (sin natalidad) baja suavemente por longevidad; la de $TBP_A$ baja mucho más y tiene la caída de natalidad encima. La diferencia entre ambas es el aporte de la natalidad.
* **Todavía no se puede afirmar:** niveles absolutos ($\bar A$ incierto), umbral "TBP=1", ni proyecciones más allá de 2024 basadas en natalidad ONU.

## 6. Fe de erratas (02-10-2026)

Las secciones 1 a 5 de arriba se dejan **tal como se presentaron originalmente**. Una revisión posterior detectó los puntos siguientes; se corrigen acá, al final, para que quede registro de la evolución del trabajo.

| # | Dónde | Decía | Corrección | Por qué |
|---|---|---|---|---|
| 1 | Sección 5, *Longevidad* | $E_{65}$ pasa de 17,9 a ~24,6 | $E_{65}$ va de **16,1** (cohorte 1956) a 24,6 (cohorte 2035); 17,9 corresponde a la cohorte 1950 | La cohorte 1956 cumple 65 en 2021 y $E_r$ sale de la tabla de *período* de ese año (COVID): pico atípico, no un error del cálculo |
| 2 | Sección 5, *curva de $TBP_B$* | "baja suavemente"; (en el resumen se citaron rangos 0,91→0,66 y 0,02–0,06) | $TBP_B$ va de **0,66 a 1,01**; $TBP_A$ de **0,018 a 0,066**; $TBP_C$ de 0,43 a 1,13 años. El máximo de los tres está en la cohorte 1956 | El resumen miró solo los extremos y omitió ese pico |
| 3 | Introducción y gráfico (*"Ā = 30, techo"*) | 30 años = techo legal | 30 años es el **requisito mínimo** para jubilarse, **no un máximo**: un promedio con ceros puede quedar muy por debajo | Error de concepto |
| 4 | Sección 5, natalidad | (no se mencionaba) | $N_t$ salta de 413.135 (DEIS 2024) a 508.067 (ONU 2025) | Es el cambio de fuente, no un hecho demográfico: el rebote posterior es un artefacto del supuesto ONU |

La celda siguiente **recalcula** esas cifras desde los datos, para que cualquiera pueda verificarlas.

In [ ]:
c = d.set_index("cohorte")
print(f"E_65: mín = {d.E65.min():.2f} (cohorte {int(d.loc[d.E65.idxmin(), 'cohorte'])}) | máx = {d.E65.max():.2f} (cohorte {int(d.loc[d.E65.idxmax(), 'cohorte'])}) | cohorte 1950 = {c.E65[1950]:.2f}")
print(f"TBP_B (Ā=30): {d.TBP_B_legal.min():.3f} a {d.TBP_B_legal.max():.3f} (máx. en la cohorte {int(d.loc[d.TBP_B_legal.idxmax(), 'cohorte'])})")
print(f"TBP_A (Ā=30): {d.TBP_A_legal.min():.4f} a {d.TBP_A_legal.max():.4f}")
print(f"TBP_C (Ā=30, años): {d.TBP_C_legal.min():.3f} a {d.TBP_C_legal.max():.3f}")
print(f"N_t usado: cohorte 2024 = {c.N_t[2024]:,.0f} ({c.fuente_N[2024]}) -> cohorte 2025 = {c.N_t[2025]:,.0f} ({c.fuente_N[2025]})")

### 7.7 Solución 1 — Sensibilidad por bandas del parámetro $\bar A$

**Opción 1 de la propuesta del grupo.** En vez de fijar $\bar A$ en un único valor, corremos el modelo para $\bar A \in \{15, 20, 25, 30\}$ años y graficamos una **banda de incertidumbre**.

Contexto: hoy $\bar A$ tiene dos escenarios: 14,2 (observado ANSES, **sesgado a la baja**: ANSES cuenta aportes solo desde jul-1994 y a personas con ≥12 meses de aporte) y 30 (requisito legal de referencia: es el mínimo para jubilarse, no un máximo). La verdad está entre ambos, pero no sabemos dónde.

$$TBP_A = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,E_r\,\rho},\qquad TBP_C = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,\rho}\ \text{(años)},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}\ \text{(cohorte pura)}$$

Constantes: $\tau = 0{,}2177$ (11 % personal + 10,77 % patronal; Anuario ANSES p. 39) y $\rho = 0{,}403$ (tasa de sustitución ANSES, dic-2023, p. 91). Todo lo demás sale del CSV `tbp_argentina_cohortes_1950_2035.csv`; no se inventa ningún dato.

> **Qué NO hace este notebook:** no corrige el sesgo de $\bar A$ ni lo estima. Es una forma honesta de *mostrar* la incertidumbre de nivel. La estimación va por la Opción 2 (calibración con EPH) y la Opción 3 ($\bar A_t$ dinámico).

### 7.2 Parámetros y cálculo para cada escenario de $\bar A$

In [ ]:
OUT = globals().get("OUT", DATA)   # carpeta de salidas del notebook
TAU = 0.11 + 0.1077     # Anuario ANSES p.39
RHO = 0.403             # Anuario ANSES p.91 (dic-2023), constante
A_ESCENARIOS = [15, 20, 25, 30]     # parametrizable
CORTE = 2024.5          # DEIS observado (<=2024) vs ONU proyectado (>=2025)

# N_t * S65 / J = D (factor demografico, ya en el CSV); lo reconstruimos desde las columnas base
d["D_calc"] = d.N_t * d.S65 / d.J
assert np.allclose(d.D_calc, d.D)

filas = []
for A in A_ESCENARIOS:
    t = pd.DataFrame({"cohorte": d.cohorte, "anio_65": d.anio_65, "fuente_N": d.fuente_N, "A": A})
    t["TBP_A"] = d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    t["TBP_C"] = d.N_t * d.S65 * A * TAU / (d.J * RHO)
    t["TBP_B"] = A * TAU / (d.E65 * RHO)
    for v in ["TBP_A", "TBP_B", "TBP_C"]:
        base = t.loc[t.cohorte == 1950, v].iloc[0]
        t[v + "_idx"] = 100 * t[v] / base          # indice base cohorte 1950 = 100
    filas.append(t)
sens = pd.concat(filas, ignore_index=True)
sens = sens.rename(columns={"A": "A_barra"})
sens.head()

### 7.3 Test de control (asserts)

1. **Consistencia con el CSV existente:** nuestro cálculo debe reproducir `TBP_*_legal` (Ā = 30) y `TBP_*_obs` (Ā = 14,2) del CSV.
2. **Linealidad en $\bar A$:** $TBP(2\bar A) = 2\cdot TBP(\bar A)$.
3. **Invarianza del índice normalizado:** el índice base 1950 = 100 es idéntico en todos los escenarios.

In [ ]:
def tbp(A, tipo):
    if tipo == "A": return d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    if tipo == "C": return d.N_t * d.S65 * A * TAU / (d.J * RHO)
    if tipo == "B": return A * TAU / (d.E65 * RHO)

# 1) reproduce el CSV existente
for tipo in "ABC":
    assert np.allclose(tbp(30.0, tipo), d[f"TBP_{tipo}_legal"])
    assert np.allclose(tbp(14.2, tipo), d[f"TBP_{tipo}_obs"])
print("OK 1) reproduce TBP_*_legal (A=30) y TBP_*_obs (A=14,2) del CSV")

# 2) linealidad
for tipo in "ABC":
    for A in [7.5, 10, 12.5, 15]:
        assert np.allclose(tbp(2 * A, tipo), 2 * tbp(A, tipo))
print("OK 2) linealidad: TBP(2A) = 2*TBP(A) para A, B y C")

# 3) indice normalizado identico entre escenarios
for v in ["TBP_A_idx", "TBP_B_idx", "TBP_C_idx"]:
    piv = sens.pivot(index="cohorte", columns="A_barra", values=v)
    ref = piv.iloc[:, 0]
    for c in piv.columns:
        assert np.allclose(piv[c], ref)
print("OK 3) el indice normalizado (base 1950=100) es identico entre escenarios")

### 7.4 Hallazgo clave: $\bar A$ cambia el **nivel**, no la **historia**

Como $\bar A$ entra en el numerador como factor multiplicativo y no depende de la cohorte $t$ (en este modelo es una constante), el TBP es **lineal homogéneo** en $\bar A$:

$$TBP_C(t;\bar A) = \bar A\cdot\underbrace{\frac{N_t(1-\mu_t)\,\tau}{J_{t+65}\,\rho}}_{k_C(t)\ \text{(no depende de }\bar A)}$$

Entonces, para dos escenarios cualesquiera $\bar A_1,\bar A_2$ y dos cohortes $t,s$:

$$\frac{TBP_C(t;\bar A_1)}{TBP_C(s;\bar A_1)}=\frac{\bar A_1\,k_C(t)}{\bar A_1\,k_C(s)}=\frac{k_C(t)}{k_C(s)}=\frac{\bar A_2\,k_C(t)}{\bar A_2\,k_C(s)}=\frac{TBP_C(t;\bar A_2)}{TBP_C(s;\bar A_2)}$$

$\bar A$ se cancela. Lo mismo vale para $TBP_A$ ($k_A=k_C/E_r$) y $TBP_B$. Por lo tanto:

* El **índice normalizado** (cohorte 1950 = 100) es **idéntico** para cualquier $\bar A$: $\text{Idx}(t)=100\,k(t)/k(1950)$.
* Las **variaciones porcentuales**, los **cocientes entre cohortes**, los **puntos de inflexión** y las **tendencias relativas** no dependen de $\bar A$.
* Lo que sí cambia con $\bar A$ es el **nivel**: la curva se estira verticalmente en un factor $\bar A_2/\bar A_1$ (entre $\bar A=15$ y $\bar A=30$, un factor 2).

**Consecuencia para la presentación:** la incertidumbre sobre $\bar A$ es incertidumbre de *nivel*, no de *comportamiento*. La historia (caída por natalidad y longevidad, ver abajo) es robusta; el valor absoluto, no.

> **Advertencia sobre la parte proyectada:** a partir de la cohorte 2025 los nacimientos $N_t$ son proyección ONU, no dato DEIS. La ONU proyecta ~507 mil nacimientos en 2024 vs ~413 mil observados por DEIS; el **rebote posterior a 2024 es un artefacto del supuesto ONU**, no un dato. No debe leerse como recuperación.
>
> **Sobre el nivel:** no interpretamos "TBP = 1" como umbral fiscal; el nivel depende de $\bar A$, de $\tau$ y de $\rho$ constantes, y no de un análisis de sostenibilidad.

### 7.5 Gráficos

In [ ]:
COL = plt.cm.viridis(np.linspace(0.1, 0.85, len(A_ESCENARIOS)))

def banda(ax, var, titulo, ylabel):
    piv = sens.pivot(index="cohorte", columns="A_barra", values=var)
    ax.fill_between(piv.index, piv.min(axis=1), piv.max(axis=1), color="tab:blue", alpha=0.15,
                    label=f"banda min-max (A = {min(A_ESCENARIOS)} a {max(A_ESCENARIOS)})")
    for c, A in zip(COL, A_ESCENARIOS):
        ax.plot(piv.index, piv[A], color=c, lw=1.6, label=f"A = {A}")
    ax.axvline(CORTE, color="gray", ls="--", lw=1)
    ax.text(CORTE + 0.5, ax.get_ylim()[1] * 0.97, "2024/2025: DEIS | ONU proyectado",
            fontsize=7.5, color="gray", va="top")
    ax.set_title(titulo, fontsize=10); ax.set_ylabel(ylabel); ax.set_xlabel("año de nacimiento de la cohorte")
    ax.grid(alpha=.3); ax.legend(fontsize=7.5, loc="upper right", bbox_to_anchor=(1, 0.88))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
banda(ax[0], "TBP_C", "TBP_C (versión recomendada, en años)\nbanda de incertidumbre de NIVEL", "TBP_C (años)")
banda(ax[1], "TBP_A", "TBP_A (fórmula original, con $E_r$)\nbanda de incertidumbre de NIVEL", "TBP_A")

# Panel (b): indice normalizado
for c, A in zip(COL, A_ESCENARIOS):
    ss = sens[sens.A_barra == A]
    ax[2].plot(ss.cohorte, ss.TBP_C_idx, color=c, lw=3.5 - 0.7 * A_ESCENARIOS.index(A), alpha=.8, label=f"A = {A}")
ax[2].axvline(CORTE, color="gray", ls="--", lw=1)
ax[2].axhline(100, color="gray", ls=":", lw=.8)
ax[2].set_title("Índice normalizado de TBP_C (cohorte 1950 = 100)\nlas 4 curvas se superponen: Ā no cambia la forma", fontsize=10)
ax[2].set_xlabel("año de nacimiento de la cohorte"); ax[2].set_ylabel("índice (1950 = 100)")
ax[2].grid(alpha=.3); ax[2].legend(fontsize=7.5)
ax[2].text(CORTE + 0.5, ax[2].get_ylim()[1] * 0.97, "post-2024: supuesto ONU\n(rebote = artefacto)", fontsize=7.5, color="gray", va="top")

fig.suptitle("Sensibilidad al parámetro Ā (años de aporte promedio): cambia el nivel, no la historia", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"{OUT}/tbp_sensibilidad_A.png", dpi=130)
plt.show()

### 7.6 Resultados: rango de $TBP_C$ por escenario en cohortes seleccionadas

In [ ]:
sel = [1950, 1975, 1990, 2000, 2010, 2024]
tabla = sens[sens.cohorte.isin(sel)].pivot(index="cohorte", columns="A_barra", values="TBP_C").round(3)
tabla["max/min"] = (tabla[max(A_ESCENARIOS)] / tabla[min(A_ESCENARIOS)]).round(2)
display(tabla)

idx = sens[(sens.A_barra == A_ESCENARIOS[0]) & sens.cohorte.isin(sel)].set_index("cohorte")["TBP_C_idx"].round(1)
print("Índice TBP_C (1950=100), igual en todos los escenarios:")
print(idx.to_string())

### 7.7 Exportar

`tbp_sensibilidad_A.csv`: una fila por cohorte × escenario, con TBP_A, TBP_B, TBP_C en nivel y como índice base 1950 = 100.

In [ ]:
sens.to_csv(f"{OUT}/tbp_sensibilidad_A.csv", index=False)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.csv", sens.shape)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.png")

### 7.8 Lectura y límites

* **Robusto (no depende de Ā):** forma de la curva, caída relativa entre cohortes, tendencias. Se comunica con el índice normalizado.
* **Incierto (sí depende de Ā):** el nivel. Se comunica con la banda 15–30 (factor 2 entre extremos).
* **No resuelto:** el sesgo de $\bar A$. Las cuatro cifras 15/20/25/30 son una grilla ilustrativa, no una distribución de probabilidad: la banda no es un intervalo de confianza.
* **Post-2024:** el rebote es un artefacto del supuesto ONU de nacimientos.
* **Próximos pasos:** Opción 2 (calibrar $\bar A$ con EPH) para acotar el nivel; Opción 3 ($\bar A_t$ dinámico por cohorte) que, a diferencia de esta, **sí** puede cambiar la forma de la curva.

### 8.8 Solución 2 — Calibración de Ā con la EPH (método; sin datos reales todavía)

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">
DATOS SINTÉTICOS — no son Argentina<br>(las celdas de prueba usan un dataset inventado; sirven solo para verificar el código)
</h1>

**Objetivo.** Estimar Ā (años de aporte promedio por persona de la cohorte, con ceros incluidos) como

Ā_c = Σ_{a=18}^{64} S_c(a) · d_c(a)

donde S_c(a) = P(sobrevivir hasta la edad a) y d_c(a) = P(aportar a la edad a | vivo), aproximada con la **tasa de empleo registrado** de la EPH tratada como pseudo-panel por año de nacimiento (cohorte = ANO4 − CH06) y edad.

**Estado:** este notebook NO produce ningún valor de Ā para Argentina hasta que se coloquen los microdatos reales de la EPH en `datos/eph/` (ver `datos/eph/README.md` y `docs/calibracion_A_EPH.md`). Sin esos archivos termina limpio con el mensaje "faltan datos".

Estructura: (1) configuración y funciones, (2) prueba con datos sintéticos + asserts, (3) prueba del lector de archivos, (4) corrida real (solo si hay datos).

In [ ]:
# ============ 1. CONFIGURACIÓN Y FUNCIONES ============
import os, re, io, glob, zipfile, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)          # 47 edades simples
RUTAS_CANDIDATAS = ["datos/eph", "../datos/eph", "../../datos/eph"]
RUTAS_S65 = ["datos/procesados/tbp_argentina_cohortes_1950_2035.csv",
             "../datos/procesados/tbp_argentina_cohortes_1950_2035.csv"]
MIN_N_CELDA = 20        # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3  # mínimo de edades observadas para estimar el efecto de una cohorte
GOMPERTZ_B = 0.085      # pendiente de la mortalidad adulta (supuesto; ver docs)
# Regla para cuentapropistas/patrones (la base individual NO trae una variable inequívoca de
# aporte de independientes: a verificar contra el diseño de registro). Opciones:
#   "solo_asalariados": independientes cuentan como NO aportantes (cota inferior)
#   "pp07i":            independientes aportan si PP07I == 1 (uso a verificar; ver docs)
#   "escenario":        independientes aportan con probabilidad IND_SHARE (supuesto explícito)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}

def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))

# ---------- (a) lectura de archivos EPH ----------
def find_eph_files(rutas=RUTAS_CANDIDATAS):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip) en las rutas candidatas."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(r"usu_individual.*\.(txt|csv)$", b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(r"usu_individual.*\.(txt|csv)$", m.lower()):
                            out.append({"path": p, "member": m})
    return out

def _read_csv_flex(handle_or_path):
    df = pd.read_csv(handle_or_path, sep=";", usecols=lambda c: c.strip().upper() in NEEDED,
                     low_memory=False, encoding="latin-1")
    return df

def read_individual(src):
    """Lee UNA base individual trimestral (solo columnas necesarias). Devuelve DataFrame con columnas en MAYÚSCULAS."""
    if src["member"] is None:
        df = _read_csv_flex(src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            df = _read_csv_flex(io.TextIOWrapper(f, encoding="latin-1"))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = {"ANO4", "CH06", "ESTADO", "PONDERA"} - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df

def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)

# ---------- (b) pseudo-panel: densidad por edad x cohorte ----------
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Filtra 18-64 con entrevista individual realizada y construye 'reg' (prob. de estar aportando).
    Asalariado (CAT_OCUP==3): reg = (PP07H==1). Independiente (1,2): según ind_rule. Resto: 0."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    reg[asal.values] = (d.loc[asal, "PP07H"] == 1).astype(float).values   # Ns/Nr (9) o 2 => no registrado
    if ind_rule == "pp07i":
        reg[indep.values] = (d.loc[indep, "PP07I"] == 1).astype(float).values
    elif ind_rule == "escenario":
        if ind_share is None or not (0 <= ind_share <= 1):
            raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
        reg[indep.values] = float(ind_share)
    elif ind_rule != "solo_asalariados":
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)     # error de +-1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]

def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    g = g[g["n"] >= min_n].reset_index(drop=True)
    return g

# ---------- modelo logit aditivo cohorte + edad (supuesto: perfil por edad estable entre cohortes) ----------
def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(1 / (1 + np.exp(-eta)), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}

def _sig(x): return 1 / (1 + np.exp(-x))

def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes sin datos suficientes (jóvenes o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)

def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo."""
    f = fit["f"].reindex(EDADES)
    f = f.interpolate(limit_direction="both")           # edades sin ninguna observación: interpola/extiende el perfil
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)

# ---------- (c) supervivencia y Ā ----------
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64, Gompertz con pendiente b calibrado para que S(65)=s65 (por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))

def abar_por_cohorte(dens, s65):
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S

def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """df individual EPH -> tabla por cohorte. Idéntico para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)})
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}

def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera sum S(a) (densidad=1)"
    # monotonía: con la misma densidad, mayor S65 => mayor A_bar
    s_alta = (t["S65"] + 0.02).clip(upper=0.999)
    a2, _ = abar_por_cohorte(dens, s_alta)
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True
print("Funciones cargadas.")

### 8.2 Prueba con DATASET SINTÉTICO

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">DATOS SINTÉTICOS — no son Argentina</h1>

Se simula una EPH "de juguete" (columnas con los mismos nombres que la base individual de INDEC) con una verdad conocida (efecto de cohorte + perfil por edad + una pequeña oscilación de período), para comprobar que el pipeline recupera Ā. **Los números de esta sección no dicen nada sobre Argentina.**

In [ ]:
# ============ 2. DATASET SINTÉTICO + ASSERTS ============
banner("DATOS SINTÉTICOS — no son Argentina")

def f_edad_sintetica(a):  # perfil por edad inventado (logit)
    return 1.2 * np.exp(-((a - 40) / 18.0) ** 2) - 1.0

def alpha_sintetico(c):   # efecto de cohorte inventado (logit)
    return -0.2 + 0.008 * (np.asarray(c) - 1970)

def periodo_sintetico(y):
    return 0.10 * np.sin((np.asarray(y) - 2003) / 3.0)

def s65_sintetico(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)

def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    rng = np.random.default_rng(seed)
    filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(alpha_sintetico(y - a) + f_edad_sintetica(a) + periodo_sintetico(y))
                u = rng.random(n_por_edad)
                reg = u < p
                otro = ~reg
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))   # no registrados: 45% ocupados, resto inactivos
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)  # parte son cuentapropistas
                pp07h = np.where(cat == 3, np.where(reg, 1, 2), np.nan)
                pp07i = np.where(cat == 2, 2, np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad),
                                           "CH06": a, "ESTADO": estado, "CAT_OCUP": cat, "PP07H": pp07h,
                                           "PP07I": pp07i, "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)

df_sint = generar_eph_sintetica()
print("Filas sintéticas:", len(df_sint))
cohortes_sint = np.arange(1940, 2036)
s65_sint = pd.Series(s65_sintetico(cohortes_sint), index=cohortes_sint)
res_s = pipeline(df_sint, s65_sint, cohortes=cohortes_sint)
t = res_s["tabla"]

# verdad conocida
dens_true = pd.DataFrame(_sig(alpha_sintetico(cohortes_sint)[:, None] + f_edad_sintetica(EDADES)[None, :]),
                         index=cohortes_sint, columns=EDADES)
a_true, _ = abar_por_cohorte(dens_true, s65_sint)
t["A_bar_verdad_sintetica"] = a_true
t["error"] = t["A_bar"] - t["A_bar_verdad_sintetica"]

# --- asserts de control ---
chequeos(res_s)
obs = t[t["n_edades_obs"] >= MIN_EDADES_COHORTE]
mae = obs["error"].abs().mean()
print(f"[sintético] error absoluto medio de Ā en cohortes observadas: {mae:.3f} años")
assert mae < 1.5, "el pipeline no recupera la verdad sintética (tolerancia 1,5 años; el efecto de período sintético se confunde parcialmente con cohorte/edad: problema APC)"
fut = t[t["n_edades_obs"] < MIN_EDADES_COHORTE]
print(f"[sintético] error medio en cohortes SIN datos (extrapoladas con 'ultimas_k'): {fut['error'].mean():+.3f} años "
      "(sesgo esperado si hay tendencia entre cohortes: es el costo del supuesto de extrapolación)")
# monotonía en S65 a igual densidad (test directo)
a_lo, _ = abar_por_cohorte(res_s["dens"], s65_sint - 0.05)
a_hi, _ = abar_por_cohorte(res_s["dens"], s65_sint + 0.02)
assert np.all(a_hi.values > a_lo.values), "Ā debería crecer con S65"
# densidad cero => Ā cero ; densidad 1 => Ā = sum S(a) <= 47
z, _ = abar_por_cohorte(res_s["dens"] * 0.0, s65_sint); assert np.allclose(z, 0)
u, S1 = abar_por_cohorte(res_s["dens"] * 0.0 + 1.0, s65_sint); assert np.all(u <= 47) and np.all(u > 0)
print("Todos los asserts sintéticos OK.")

# gráficos
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c, col in [(1950, "C0"), (1970, "C1"), (1990, "C2"), (2000, "C3")]:
    cc = res_s["cells"][res_s["cells"]["cohorte"] == c]
    ax[0].scatter(cc["edad"], cc["dens"], s=10, color=col, alpha=.6)
    ax[0].plot(EDADES, res_s["dens"].loc[c].values, color=col, label=f"cohorte {c}")
ax[0].set_xlabel("edad"); ax[0].set_ylabel("densidad d(a)"); ax[0].legend(); ax[0].set_title("SINTÉTICO: puntos observados vs modelo")
ax[1].plot(t.index, t["A_bar"], label="estimado"); ax[1].plot(t.index, t["A_bar_verdad_sintetica"], "--", label="verdad sintética")
ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("Ā (años)"); ax[1].legend()
ax[1].set_title("SINTÉTICO: Ā por cohorte")
fig.suptitle("DATOS SINTÉTICOS — no son Argentina", color="red", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()
banner("FIN PRUEBA SINTÉTICA — estos números NO son Argentina")

### 8.3 Prueba del lector de archivos (con un archivo de juguete en formato INDEC)

Escribe un subconjunto del dataset **sintético** en una carpeta temporal con el nombre `usu_individual_T100.txt` (separador `;`), y verifica que `find_eph_files` + `read_all` + `pipeline` dan lo mismo que con el DataFrame en memoria. No toca `datos/eph/`.

In [ ]:
banner("DATOS SINTÉTICOS — prueba del lector, no son Argentina")
with tempfile.TemporaryDirectory() as tmp:
    sub = df_sint[df_sint["ANO4"].isin([2010, 2011, 2012, 2013])].copy()
    sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]  # mezcla de mayúsculas/minúsculas
    sub.to_csv(os.path.join(tmp, "usu_individual_T100.txt"), sep=";", index=False)
    srcs = find_eph_files([tmp]); assert len(srcs) == 1
    leido = read_all(srcs)
    assert set(NEEDED) <= set(leido.columns)
    assert len(leido) == len(sub)
    # zip con el nombre típico de INDEC
    zp = os.path.join(tmp, "EPH_usu_1_Trim_2012_txt.zip")
    with zipfile.ZipFile(zp, "w") as z:
        z.write(os.path.join(tmp, "usu_individual_T100.txt"), "usu_individual_T112.txt")
    os.remove(os.path.join(tmp, "usu_individual_T100.txt"))
    srcs2 = find_eph_files([tmp]); assert len(srcs2) == 1 and srcs2[0]["member"] is not None
    assert len(read_all(srcs2)) == len(sub)
print("Lector OK (txt y zip) con datos sintéticos.")
# find_eph_files sobre una carpeta inexistente => lista vacía, sin error
assert find_eph_files(["/ruta/que/no/existe"]) == []

### 8.4 Corrida con datos REALES (solo si están en `datos/eph/`)

Si no hay archivos, el notebook termina limpio con "faltan datos". Si los hay, aplica el mismo `pipeline` y guarda `salidas_tbp/abar_eph_por_cohorte.csv` (carpeta ignorada por git). Las cifras de referencia 14,2 (ANSES, observado, sesgado a la baja) y 30 (requisito legal de referencia, no un máximo) se muestran solo como contexto; **no son objetivo de calibración**.

In [ ]:
# ============ 4. CORRIDA REAL ============
srcs_reales = find_eph_files(RUTAS_CANDIDATAS)
HAY_DATOS = len(srcs_reales) > 0

if not HAY_DATOS:
    print("=" * 78)
    print("FALTAN DATOS: no se encontraron bases individuales de la EPH en", RUTAS_CANDIDATAS)
    print("Qué hacer: bajar las bases individuales (usu_individual_T*.txt o EPH_usu_*_txt.zip) desde INDEC")
    print("y dejarlas en datos/eph/ (ver datos/eph/README.md). No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    s65_path = next((p for p in RUTAS_S65 if os.path.exists(p)), None)
    if s65_path is None:
        print("FALTAN DATOS: no se encontró tbp_argentina_cohortes_1950_2035.csv (S65 por cohorte).")
        HAY_DATOS = False
if HAY_DATOS:
    print(f"Archivos EPH encontrados: {len(srcs_reales)}")
    df_real = read_all(srcs_reales)
    print("Años cubiertos:", int(df_real["ANO4"].min()), "-", int(df_real["ANO4"].max()), "| filas:", len(df_real))
    s65_real = pd.read_csv(s65_path).set_index("cohorte")["S65"]
    coh_real = s65_real.index
    escenarios = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
                  "pp07i": dict(ind_rule="pp07i"),
                  "indep_50pct (supuesto)": dict(ind_rule="escenario", ind_share=0.5)}
    salida = {}
    for nombre, kw in escenarios.items():
        r = pipeline(df_real, s65_real, cohortes=coh_real, **kw)
        chequeos(r)
        salida[nombre] = r["tabla"]["A_bar"]
        salida["n_edades_obs"] = r["tabla"]["n_edades_obs"]
    out = pd.DataFrame(salida)
    os.makedirs("salidas_tbp", exist_ok=True)
    out.to_csv("salidas_tbp/abar_eph_por_cohorte.csv")
    print(out.loc[[c for c in (1960, 1970, 1980, 1990, 2000) if c in out.index]].round(2))
    print("Referencias de contexto (NO objetivo): ANSES observado 14,2 (sesgado a la baja); requisito legal 30 (techo).")